# Fase 2 — Obtención, limpieza y transformación de datos

**Proyecto ABP — Ciencia de Datos Reproducible (MCDI500) · Grupo 3**

Integrantes: Rodrigo Chinchón Ayala · Sergio Fernández Almonacid · Pablo Villalobos González

Notebook ejecutable del **pipeline de preprocesamiento**. Implementa la obtención, exploración inicial, limpieza, transformación y validación técnica del conjunto de datos, sobre el entorno reproducible configurado en la Fase 1. El código operativo reside en `F2/src/preprocessing.py` (modularidad); aquí se orquesta y documenta cada etapa.

## 1. Objetivos de la Fase 2

**Objetivo general.** Implementar un pipeline reproducible que cargue, perfile, limpie, transforme y valide el dataset del proyecto, asegurando calidad y consistencia para el análisis posterior.

**Objetivos específicos.**
1. Obtener el dataset y realizar una exploración inicial (tipos, nulos, duplicados).
2. Aplicar una limpieza exhaustiva: duplicados, manejo de NA y normalización textual.
3. Transformar variables (casting, normalización MinMax y One Hot Encoding).
4. Validar el dataset resultante y verificar el comportamiento ante casos límite.
5. Persistir el dataset procesado de forma trazable.

**Dataset.** Cartera de clientes/operaciones (segmento, región, antigüedad, monto, mora, canal, estado, score de riesgo). Se eligió un conjunto pequeño con defectos deliberados (nulos, duplicados, texto inconsistente) para evidenciar cada técnica de limpieza.

## 2. Configuración e importación de módulos

Se importan las funciones del pipeline desde `F2/src/preprocessing.py`. El bloque resuelve la ruta raíz del proyecto para que el notebook funcione tanto desde `F2/notebooks/` como desde la raíz.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parents[1]
sys.path.append(str(ROOT / 'F2' / 'src'))
from preprocessing import (  # noqa: E402
    load_dataset, profile_dataset, clean_dataset,
    transform_dataset, validate_dataset, run_pipeline,
)

RAW = ROOT / 'F2' / 'data' / 'raw' / 'dataset_base.csv'
OUT = ROOT / 'F2' / 'data' / 'processed' / 'dataset_procesado.csv'
print('Pandas :', pd.__version__)
print('Entrada:', RAW)

## 3. Obtención y exploración inicial

Se carga el dataset crudo y se perfila su estado: dimensiones, tipos, nulos y duplicados. Esta es la línea base contra la que se comparará el resultado de la limpieza.

In [ ]:
df_raw = load_dataset(RAW)
print('Dimensiones:', df_raw.shape)
print('Duplicados exactos:', df_raw.duplicated().sum())
display(df_raw.head())
display(profile_dataset(df_raw))

## 4. Limpieza

**Criterios aplicados** (cada uno justificado en la sección 8):
1. Eliminación de duplicados exactos.
2. Normalización textual de categóricas (espacios y capitalización).
3. Casting seguro de numéricas (texto inválido → NaN).
4. Imputación de numéricas con la **mediana**.
5. Relleno de categóricas no informadas con `'No informado'`.

In [ ]:
df_clean = clean_dataset(df_raw)
print('Dimensiones tras limpieza:', df_clean.shape)
display(df_clean.head())
display(profile_dataset(df_clean))
print('Validación:', validate_dataset(df_clean))

## 5. Transformación

Se normalizan las variables numéricas con **MinMax** (escala 0–1) y se aplica **One Hot Encoding** a las variables nominales, dejando el dataset apto para análisis o modelado.

In [ ]:
df_transformed = transform_dataset(df_clean)
print('Dimensiones tras transformación:', df_transformed.shape)
display(df_transformed.head())
print('Validación:', validate_dataset(df_transformed))

## 6. Validación técnica y verificación (casos normales, límite y excepciones)

La rúbrica exige verificar el código bajo distintos escenarios. Se prueban: (a) el caso normal del pipeline, (b) casos límite como una columna constante o un DataFrame de una fila, y (c) el manejo de excepciones ante una ruta inexistente.

In [ ]:
# (a) CASO NORMAL: el dataset limpio no tiene nulos ni duplicados
v = validate_dataset(df_clean)
assert v['sin_nulos'], 'Quedan nulos tras la limpieza'
assert v['sin_duplicados'], 'Quedan duplicados tras la limpieza'
print('[OK] Caso normal: dataset limpio sin nulos ni duplicados')

# (b) CASO LÍMITE 1: columna constante no debe romper la normalización MinMax
df_const = df_clean.copy()
df_const['dias_mora'] = 0  # rango cero
out_const = transform_dataset(df_const)
assert out_const['dias_mora'].notna().all(), 'La columna constante generó NaN'
print('[OK] Caso límite 1: columna constante manejada sin división por cero')

# (c) CASO LÍMITE 2: DataFrame de una sola fila
una_fila = clean_dataset(df_raw.head(1))
assert len(una_fila) == 1, 'El pipeline no maneja una sola fila'
print('[OK] Caso límite 2: pipeline opera con una sola fila')

# (d) EXCEPCIÓN: ruta inexistente debe lanzar FileNotFoundError
try:
    load_dataset(ROOT / 'F2' / 'data' / 'raw' / 'no_existe.csv')
    print('[FALLO] No se lanzó la excepción esperada')
except FileNotFoundError:
    print('[OK] Excepción: FileNotFoundError capturada correctamente')

## 7. Pipeline completo y persistencia

Se ejecuta el pipeline de extremo a extremo y se guarda el dataset procesado en `F2/data/processed/`, dejando trazable la salida.

In [ ]:
clean, transformed, validation = run_pipeline(RAW, OUT)
print('Validación final:', validation)
assert validation['sin_nulos'], 'El dataset procesado mantiene valores nulos.'
assert validation['sin_duplicados'], 'El dataset procesado mantiene duplicados.'
assert OUT.exists(), 'No se generó el archivo procesado.'
print('Dataset procesado guardado en:', OUT)

## 8. Justificación técnica de las transformaciones

- **Imputación con la mediana** (no la media): la mediana es robusta ante valores atípicos. En variables como `monto_operacion` o `dias_mora`, unos pocos valores extremos distorsionarían la media; la mediana preserva mejor la tendencia central y evita introducir sesgos en registros imputados.
- **Normalización MinMax** (escala 0–1): las variables numéricas tienen rangos muy dispares (p. ej. `monto_operacion` en millones frente a `score_riesgo` entre 0 y 1). Llevarlas a una escala común evita que las de mayor magnitud dominen análisis de distancia o modelos sensibles a la escala, sin alterar la forma de su distribución.
- **One Hot Encoding** de nominales (`segmento`, `region`, `canal`, `estado`): son categorías sin orden intrínseco. Codificarlas como enteros introduciría un orden falso; One Hot crea columnas binarias independientes, representación correcta para variables nominales.
- **Normalización textual** (espacios y capitalización): unifica valores que representan lo mismo escrito distinto (`' web '`, `'EJECUTIVO'`), evitando categorías duplicadas espurias en el encoding.

**Relación con los objetivos.** Estas decisiones garantizan un dataset íntegro, consistente y comparable, condición necesaria para el análisis y modelado de fases posteriores.